# 02 - Preprocessing & Playlist Construction

This notebook does two things:

1. Cleans the raw track catalog (drop duplicates/nulls, normalize types).
2. Builds a **playlist -> track** interaction table the collaborative-filtering
   model can train on.

### Why we build playlists instead of loading them

The real interaction data for this kind of recommender is Spotify's Million
Playlist Dataset (MPD): millions of real user playlists, each a list of
track URIs. It's the right training data for a production system, but it
requires registering for restricted access through Spotify/AIcrowd's
research program and is ~33GB, so it isn't something this notebook can pull
in automatically.

What we do instead: keep the **track catalog 100% real** (the 114k tracks
from notebook 01), and construct playlists deterministically by grouping
real tracks that share a genre into fixed-size chunks, ordered by
popularity. This is a common technique when real interaction logs aren't
available, and it gives the interaction matrix a realistic shape (playlists
that are internally coherent by genre, with overlapping tracks across
playlists) without pretending it's real user behavior. **This is clearly a
simulated grouping, not real listening/playlist history** - swap in real
MPD slices in `data/raw/` and this notebook's output format stays the same,
since everything downstream only needs `pid` / `track_uri` columns.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/spotify_tracks_dataset.csv').drop(columns=['Unnamed: 0'])
df = df.dropna(subset=['track_name', 'artists', 'album_name']).reset_index(drop=True)
df = df.drop_duplicates(subset=['track_name', 'artists']).reset_index(drop=True)
df['track_uri'] = 'spotify:track:' + df['track_id'].astype(str)
print("Full cleaned catalog:", df.shape)

Full cleaned catalog: (81343, 21)


### Capping tracks per genre

The full cleaned catalog is ~81k tracks. That's realistic for a real catalog,
but the collaborative-filtering evaluation later scores *every* candidate
track for every playlist, which becomes very slow on a single CPU core at
that scale. We cap each genre to its 150 most popular tracks (still real
tracks, just a bounded, popularity-biased sample) so the rest of the
pipeline runs in a reasonable amount of time here. Drop this cap (or raise
it) if you're running on more cores / don't mind longer training and
evaluation times.

In [2]:
TRACKS_PER_GENRE_CAP = 150
df = (df.sort_values('popularity', ascending=False)
        .groupby('track_genre', group_keys=False)
        .head(TRACKS_PER_GENRE_CAP)
        .reset_index(drop=True))
print("Capped catalog used for the rest of the pipeline:", df.shape)
df[['track_uri', 'track_name', 'artists', 'album_name', 'popularity', 'track_genre']].head()

Capped catalog used for the rest of the pipeline: (16804, 21)


,track_uri,track_name,artists,album_name,popularity,track_genre
0,spotify:track:3nqQXoyQOWXiESFLlDF1hG,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100,dance
1,spotify:track:2tTmW7RDtMQtBk7m2rYeSw,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",99,hip-hop
2,spotify:track:5ww2BF9slyYgNOk37BlC4u,La Bachata,Manuel Turizo,La Bachata,98,latin
3,spotify:track:4uUG5RXrOk84mYEfFvj3cK,I'm Good (Blue),David Guetta;Bebe Rexha,I'm Good (Blue),98,dance
4,spotify:track:1IHWl5LamUGEuP4ozKQSXZ,Tití Me Preguntó,Bad Bunny,Un Verano Sin Ti,97,latin


## Building genre-based playlists

Within each genre, sort by popularity and chunk into playlists of ~25 tracks.

In [3]:
PLAYLIST_SIZE = 25
rng = np.random.RandomState(42)

playlist_rows = []
pid_counter = 0

for genre, group in df.groupby('track_genre'):
    group = group.sort_values('popularity', ascending=False).reset_index(drop=True)
    chunks = [group.iloc[i:i + PLAYLIST_SIZE] for i in range(0, len(group), PLAYLIST_SIZE)]
    for chunk in chunks:
        if len(chunk) < 3:
            continue
        playlist_name = f"{genre.title()} Mix {pid_counter}"
        for _, row in chunk.iterrows():
            playlist_rows.append({
                'pid': pid_counter,
                'playlist_name': playlist_name,
                'track_uri': row['track_uri'],
                'track_name': row['track_name'],
                'artist_name': row['artists'],
                'album_name': row['album_name'],
                'popularity': row['popularity'],
                'track_genre': row['track_genre'],
            })
        pid_counter += 1

interactions_df = pd.DataFrame(playlist_rows)
print("Playlists:", interactions_df['pid'].nunique())
print("Rows (playlist-track pairs):", len(interactions_df))
print("Unique tracks used:", interactions_df['track_uri'].nunique())
interactions_df.head()

Playlists: 674
Rows (playlist-track pairs): 16804
Unique tracks used: 16804


,pid,playlist_name,track_uri,track_name,artist_name,album_name,popularity,track_genre
0,0,Acoustic Mix 0,spotify:track:5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,82,acoustic
1,0,Acoustic Mix 0,spotify:track:1EzrEOXmMH3G43AXT1y7pA,I'm Yours,Jason Mraz,We Sing. We Dance. We Steal Things.,80,acoustic
2,0,Acoustic Mix 0,spotify:track:08MFgEQeVLF37EyZ7jcwLc,Pano,Zack Tabudlo,Pano,75,acoustic
3,0,Acoustic Mix 0,spotify:track:0IktbUcnAGrvD03AWnz3Q8,Lucky,Jason Mraz;Colbie Caillat,We Sing. We Dance. We Steal Things.,74,acoustic
4,0,Acoustic Mix 0,spotify:track:6Vc5wAMmXdKIAM7WUoEb7N,Say Something,A Great Big World;Christina Aguilera,Is There Anybody Out There?,74,acoustic


## Adding cross-playlist overlap

Real playlists share tracks (a popular song shows up in many playlists).
Our genre-chunking above gives each track exactly one playlist, which would
make the interaction matrix artificially sparse in a way real data isn't.
We add a modest number of "crossover" tracks - popular tracks that also get
added to a handful of *other* genre-adjacent playlists - to make the
collaborative-filtering signal meaningful (a purely 1-playlist-per-track
matrix gives LightFM nothing to learn from).

In [4]:
popular_tracks = df[df['popularity'] >= 40].reset_index(drop=True)
all_pids = interactions_df['pid'].unique()

crossover_rows = []
for _, row in popular_tracks.iterrows():
    extra_playlists = rng.choice(all_pids, size=min(15, len(all_pids)), replace=False)
    for pid in extra_playlists:
        playlist_name = interactions_df.loc[interactions_df['pid'] == pid, 'playlist_name'].iloc[0]
        crossover_rows.append({
            'pid': pid,
            'playlist_name': playlist_name,
            'track_uri': row['track_uri'],
            'track_name': row['track_name'],
            'artist_name': row['artists'],
            'album_name': row['album_name'],
            'popularity': row['popularity'],
            'track_genre': row['track_genre'],
        })

crossover_df = pd.DataFrame(crossover_rows)
interactions_df = pd.concat([interactions_df, crossover_df], ignore_index=True)
interactions_df = interactions_df.drop_duplicates(subset=['pid', 'track_uri']).reset_index(drop=True)

print("Rows after adding crossover popular tracks:", len(interactions_df))
print("Average tracks per playlist:", interactions_df.groupby('pid').size().mean().round(1))
print("Average playlists per track:", interactions_df.groupby('track_uri').size().mean().round(2))

Rows after adding crossover popular tracks: 225776
Average tracks per playlist: 335.0
Average playlists per track: 13.44


## Save the processed interaction table

In [5]:
interactions_df.to_csv('../data/processed/playlist_track_interactions.csv', index=False)
df.to_csv('../data/processed/tracks_clean.csv', index=False)
print("Saved:")
print(" - data/processed/playlist_track_interactions.csv", interactions_df.shape)
print(" - data/processed/tracks_clean.csv", df.shape)

Saved:
 - data/processed/playlist_track_interactions.csv (225776, 8)
 - data/processed/tracks_clean.csv (16804, 21)


The next notebook (`03_content_embeddings.ipynb`) builds text-based content vectors for every track using this cleaned data.